# Infrastructure as code with Terraform — the hands-on half

The practical companion to **`terraform_slides.html`**. Every `apply` in this notebook creates
something real and every `destroy` removes it — but using the **Docker provider**, so there is no
cloud account, no credentials and no bill.

That substitution is not a compromise. The workflow, the state file, the plan/apply cycle, the
dependency graph and the failure modes are identical to AWS; only the resource types differ.

| Part | Deck slides | What you do |
|---|---|---|
| 0 · Setup | 6 | install, sandbox, check the providers |
| 1 · First resource | 7–9 | `init`, `plan`, `apply` — a real container appears |
| 2 · State | 10 | what `terraform.tfstate` is, and why it matters more than the code |
| 3 · Change and destroy | 11–12 | plan diffs, in-place vs replace, `destroy` |
| 4 · Real structure | 13–15 | variables, outputs, dependencies, a module |
| 5 · Advanced | 16–20 | drift, import, workspaces, remote state |

Everything happens in a throwaway `tf_demo/` folder; the last cell destroys the infrastructure
and removes it.

## Step 0.1 · Install

Terraform is a single binary.

```bash
# Linux
curl -fsSL -o tf.zip https://releases.hashicorp.com/terraform/1.9.8/terraform_1.9.8_linux_amd64.zip
unzip tf.zip -d ~/.local/bin/

# macOS
brew install terraform
```

In [1]:
import os, shutil, subprocess

# find terraform wherever it is
TF = shutil.which("terraform") or os.path.expanduser("~/.local/bin/terraform")
os.environ["PATH"] = os.path.dirname(TF) + os.pathsep + os.environ["PATH"]
print(subprocess.run([TF, "version"], capture_output=True, text=True).stdout.splitlines()[0])
print("docker:", subprocess.run(["docker", "--version"], capture_output=True, text=True).stdout.strip())

Terraform v1.9.8
docker: Docker version 29.6.1, build 8900f1d


## Step 0.2 · A sandbox to work in

In [2]:
import os, shutil, pathlib

BASE = pathlib.Path.cwd()          # the folder this notebook lives in
PROJ = BASE / "tf_demo"           # a throwaway sandbox, deleted by the last cell

if PROJ.exists():
    shutil.rmtree(PROJ)            # re-running this notebook is always safe
PROJ.mkdir(parents=True)
os.chdir(PROJ)
print("working inside:", os.getcwd())

working inside: /home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/12-infrastructure/tf_demo


---
# Part 1 — One resource, end to end   ·   deck slides 7–9

Terraform is **declarative**: you describe the state you want, it works out the steps. That is the
whole difference from a shell script, and it is why running it twice is safe.

In [3]:
%%writefile main.tf
terraform {
  required_version = ">= 1.5"
  required_providers {
    docker = {
      source  = "kreuzwerker/docker"     # a real provider, against a real daemon
      version = "~> 3.0"
    }
  }
}

provider "docker" {}

# An image is a resource too -- Terraform will pull it if it is missing.
resource "docker_image" "nginx" {
  name         = "nginx:1.27-alpine"
  keep_locally = true                    # do not delete the image on destroy
}

resource "docker_container" "web" {
  name  = "tf-web"
  image = docker_image.nginx.image_id     # <- this reference IS the dependency graph

  ports {
    internal = 80
    external = 8081
  }

  labels {
    label = "managed-by"
    value = "terraform"
  }
}

Writing main.tf


A small helper so every Terraform command below shows its real output — and fails loudly rather
than quietly, which is the whole reason infrastructure code is worth writing down.

In [4]:
def tf(*args, cwd=".", check=True, quiet=False):
    """Run terraform and show what it said. Fails loudly -- a broken plan must not pass."""
    r = subprocess.run([TF, *args], cwd=cwd, capture_output=True, text=True,
                       env={**os.environ, "TF_IN_AUTOMATION": "1"})
    out = (r.stdout or "") + (r.stderr or "")
    if not quiet:
        print(out[-1800:])
    if check and r.returncode != 0:
        raise RuntimeError(f"terraform {' '.join(args)} failed")
    return out

tf("init", "-no-color")

Initializing the backend...
Initializing provider plugins...
- Finding kreuzwerker/docker versions matching "~> 3.0"...
- Installing kreuzwerker/docker v3.9.0...
- Installed kreuzwerker/docker v3.9.0 (self-signed, key ID 0DCE698927DAF8EC)
Partner and community providers are signed by their developers.
If you'd like to know more about provider signing, you can read about it here:
https://www.terraform.io/docs/cli/plugins/signing.html
Terraform has created a lock file .terraform.lock.hcl to record the provider
selections it made above. Include this file in your version control repository
so that Terraform can guarantee to make the same selections by default when
you run "terraform init" in the future.

Terraform has been successfully initialized!



'Initializing the backend...\nInitializing provider plugins...\n- Finding kreuzwerker/docker versions matching "~> 3.0"...\n- Installing kreuzwerker/docker v3.9.0...\n- Installed kreuzwerker/docker v3.9.0 (self-signed, key ID 0DCE698927DAF8EC)\nPartner and community providers are signed by their developers.\nIf you\'d like to know more about provider signing, you can read about it here:\nhttps://www.terraform.io/docs/cli/plugins/signing.html\nTerraform has created a lock file .terraform.lock.hcl to record the provider\nselections it made above. Include this file in your version control repository\nso that Terraform can guarantee to make the same selections by default when\nyou run "terraform init" in the future.\n\nTerraform has been successfully initialized!\n'

`init` downloaded the provider into `.terraform/` and wrote a **lock file** pinning its exact
version and checksums. Commit `.terraform.lock.hcl`; do not commit `.terraform/`.

In [5]:
!ls -a
print()
!head -12 .terraform.lock.hcl

.  ..  main.tf	.terraform  .terraform.lock.hcl

# This file is maintained automatically by "terraform init".
# Manual edits may be lost in future updates.

provider "registry.terraform.io/kreuzwerker/docker" {
  version     = "3.9.0"
  constraints = "~> 3.0"
  hashes = [
    "h1:EAdNh5KgGPJT5jm848MRIfNfHUVJeTBdKKcFLax5g38=",
    "zh:0ead8281830e9b9496651282235d9a139ba1b1b6ff79e395eb8c78658dc446b9",
    "zh:0f17d37d8d3872df3fb75c68b5272e0c981343f53b506a9675b4405191edd3ef",
    "zh:11d50b37323874427c6d2a08b737d3c7707c8301fdd236c94485cf2828d0b14b",
    "zh:32f6f9b847446054e2db3d72886ef2f1d1aa51a6d0dac42340b07dad18e3f28f",


## Step 1.1 · Plan before apply, always

`plan` compares three things: your configuration, the recorded state, and reality. It then tells
you what it intends to do — and nothing has happened yet.

In [6]:
plan_out = tf("plan", "-no-color")
assert "2 to add" in plan_out or "to add" in plan_out

                      = 0
      + must_run                                    = true
      + name                                        = "tf-web"
      + network_data                                = (known after apply)
      + network_mode                                = "bridge"
      + read_only                                   = false
      + remove_volumes                              = true
      + restart                                     = "no"
      + rm                                          = false
      + runtime                                     = (known after apply)
      + security_opts                               = (known after apply)
      + shm_size                                    = (known after apply)
      + start                                       = true
      + stdin_open                                  = false
      + stop_signal                                 = (known after apply)
      + stop_timeout                                = (known a

`apply` does what the plan said. Watch it create the network and the container for real.

In [7]:
apply_out = tf("apply", "-auto-approve", "-no-color")

          = true
      + restart                                     = "no"
      + rm                                          = false
      + runtime                                     = (known after apply)
      + security_opts                               = (known after apply)
      + shm_size                                    = (known after apply)
      + start                                       = true
      + stdin_open                                  = false
      + stop_signal                                 = (known after apply)
      + stop_timeout                                = (known after apply)
      + tty                                         = false
      + wait                                        = false
      + wait_timeout                                = 60

      + healthcheck (known after apply)

      + labels {
          + label = "managed-by"
          + value = "terraform"
        }

      + ports {
          + external = 8081
          + interna

Proof it was not a simulation: Docker itself now lists the container.

In [8]:
# it is not a simulation -- there is a container
!docker ps --filter name=tf-web --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}'
print()
import urllib.request, time
for _ in range(30):
    try:
        body = urllib.request.urlopen("http://127.0.0.1:8081", timeout=2).read().decode()
        print("HTTP 200 from the container Terraform created:")
        print("  ", body.splitlines()[3].strip() if len(body.splitlines()) > 3 else body[:60])
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("the container did not answer")

{.Names}   {.Status}   {.Ports}
{.Names}   {.Status}   {.Ports}

HTTP 200 from the container Terraform created:
   <title>Welcome to nginx!</title>


---
# Part 2 — State   ·   deck slides 10

`terraform.tfstate` is a JSON map from your resource names to the real objects they created. It is
the most important file in the repository, and the one people understand last.

Without it, Terraform has no idea that `docker_container.web` is that container — so it would try
to create a second one.

In [9]:
import json
state = json.load(open("terraform.tfstate"))
print(f"serial: {state['serial']}   resources: {len(state['resources'])}")
for r in state["resources"]:
    attrs = r["instances"][0]["attributes"]
    ident = attrs.get("id", "")[:12]
    print(f"  {r['type']}.{r['name']:8} -> id {ident}  ({attrs.get('name', '')})")

serial: 3   resources: 2
  docker_container.web      -> id b738ba4741d1  (tf-web)
  docker_image.nginx    -> id sha256:6769d  (nginx:1.27-alpine)


The **state file** is Terraform's record of what it built and what each thing looks like. It is
how `plan` knows the difference between "create this" and "leave it alone".

In [10]:
print(tf("state", "list", "-no-color", quiet=True))
print("--- one resource in detail ---")
print(tf("state", "show", "docker_container.web", "-no-color", quiet=True)[:700])

docker_container.web
docker_image.nginx

--- one resource in detail ---
# docker_container.web:
resource "docker_container" "web" {
    attach                                      = false
    bridge                                      = null
    command                                     = [
        "nginx",
        "-g",
        "daemon off;",
    ]
    container_read_refresh_timeout_milliseconds = 15000
    cpu_set                                     = null
    cpu_shares                                  = 0
    domainname                                  = null
    entrypoint                                  = [
        "/docker-entrypoint.sh",
    ]
    env                                         = []
    hostname                                    = "b738


### Three things to know about state

1. **It contains secrets.** A database password in a resource attribute is stored in plain text in
   the state file. That alone is the reason for remote state with encryption.
2. **It can drift from reality.** If someone changes the container by hand, the state is stale
   until the next `plan` reconciles it. That is Part 5.
3. **Losing it is bad but not fatal.** You can `terraform import` resources back, one by one. It
   is a slow afternoon rather than a catastrophe.

> **Never edit state by hand.** Use `terraform state mv`, `rm`, `import`. Hand-editing JSON is how
> a small problem becomes a big one.

---
# Part 3 — Changing things   ·   deck slides 11–12

The interesting part of Terraform is not creating; it is **changing**. Some changes happen in
place; some force the resource to be destroyed and recreated. The plan always tells you which,
and reading that distinction is the skill.

In [11]:
%%writefile main.tf
terraform {
  required_version = ">= 1.5"
  required_providers {
    docker = {
      source  = "kreuzwerker/docker"
      version = "~> 3.0"
    }
  }
}

provider "docker" {}

resource "docker_image" "nginx" {
  name         = "nginx:1.27-alpine"
  keep_locally = true
}

resource "docker_container" "web" {
  name  = "tf-web"
  image = docker_image.nginx.image_id

  ports {
    internal = 80
    external = 8081
  }

  labels {
    label = "managed-by"
    value = "terraform"
  }

  labels {
    label = "owner"
    value = "ml-platform"
  }
}

Overwriting main.tf


Change one value and plan again. Read the symbols: `+` create, `-` destroy, `~` change in place,
`-/+` destroy and recreate. The last one is the one that causes outages.

In [12]:
out = tf('plan', '-no-color')

              = 0 -> null
      - memory                                      = 0 -> null
      - memory_swap                                 = 0 -> null
        name                                        = "tf-web"
      ~ network_data                                = [
          - {
              - gateway                   = "172.17.0.1"
              - global_ipv6_prefix_length = 0
              - ip_address                = "172.17.0.2"
              - ip_prefix_length          = 16
              - mac_address               = "26:04:09:a1:ae:4a"
              - network_name              = "bridge"
                # (2 unchanged attributes hidden)
            },
        ] -> (known after apply)
      - privileged                                  = false -> null
      - publish_all_ports                           = false -> null
      ~ runtime                                     = "runc" -> (known after apply)
      ~ security_opts                               = [] -> (known afte

Look for the symbols in that plan. Terraform has a small vocabulary:

| Symbol | Means |
|---|---|
| `+` | create |
| `-` | destroy |
| `~` | update **in place** |
| `-/+` | destroy and recreate — **read this one carefully** |
| `+/-` | create then destroy (`create_before_destroy`) |

A `-/+` on a production database is how outages happen. The plan told you; somebody scrolled past it.

In [13]:
out = tf("apply", "-auto-approve", "-no-color")
!docker inspect tf-web --format '{{json .Config.Labels}}' | python3 -m json.tool

prefix_length = 0
              - ip_address                = "172.17.0.2"
              - ip_prefix_length          = 16
              - mac_address               = "26:04:09:a1:ae:4a"
              - network_name              = "bridge"
                # (2 unchanged attributes hidden)
            },
        ] -> (known after apply)
      - privileged                                  = false -> null
      - publish_all_ports                           = false -> null
      ~ runtime                                     = "runc" -> (known after apply)
      ~ security_opts                               = [] -> (known after apply)
      ~ shm_size                                    = 64 -> (known after apply)
      ~ stop_signal                                 = "SIGQUIT" -> (known after apply)
      ~ stop_timeout                                = 0 -> (known after apply)
      - storage_opts                                = {} -> null
      - sysctls                                     

Apply the same configuration twice. The second run should do **nothing** — that is what
declarative means, and it is why re-running is safe.

In [14]:
# and now the property that makes Terraform safe: converge, do not repeat
out = tf("plan", "-no-color", quiet=True)
print(out[-400:])
assert "No changes" in out, "a second plan with no config change must be empty"
print("\nIdempotent: nothing left to do. Run it a hundred times, same result.")

te... [id=sha256:6769dc3a703c719c1d2756bda113659be28ae16cf0da58dd5fd823d6b9a050eanginx:1.27-alpine]
docker_container.web: Refreshing state... [id=a16de885a532dea429db680655322d07e1b81877d77b11e16aa86056b58dbf09]

No changes. Your infrastructure matches the configuration.

Terraform has compared your real infrastructure against your configuration
and found no differences, so no changes are needed.


Idempotent: nothing left to do. Run it a hundred times, same result.


---
# Part 4 — What a real configuration looks like   ·   deck slides 13–15

Variables, outputs, and resources that depend on each other. This is where declarative pays off:
you never write the order, you write the references.

In [15]:
%%writefile variables.tf
variable "app_name" {
  description = "prefix for every resource this stack creates"
  type        = string
  default     = "churn"
}

variable "api_port" {
  description = "host port for the API"
  type        = number
  default     = 8082

  validation {
    condition     = var.api_port > 1024 && var.api_port < 65535
    error_message = "api_port must be an unprivileged port."
  }
}

variable "replicas" {
  description = "how many API containers to run"
  type        = number
  default     = 2
}

Writing variables.tf


A bigger configuration: a private network plus several containers. Nothing references anything by
name — resources point at each other, and Terraform derives the order.

In [16]:
%%writefile stack.tf
# A private network. Nothing references it by name -- resources reference the OBJECT,
# and that is what builds the dependency graph.
resource "docker_network" "app" {
  name = "${var.app_name}-net"
}

resource "docker_image" "api" {
  name         = "nginx:1.27-alpine"
  keep_locally = true
}

# count gives you N copies. Terraform tracks them as api[0], api[1], ...
resource "docker_container" "api" {
  count = var.replicas

  name  = "${var.app_name}-api-${count.index}"
  image = docker_image.api.image_id

  networks_advanced {
    # the argument here is `name`, not `network`. Getting it wrong produces two
    # errors that both point at the wrong thing: "name is required" and
    # "an argument named network is not expected here".
    name = docker_network.app.name         # depends on the network, implicitly
  }

  ports {
    internal = 80
    external = var.api_port + count.index
  }

  env = ["REPLICA=${count.index}", "APP=${var.app_name}"]
}

Writing stack.tf


`output` blocks publish values for humans and other tooling to read.

In [17]:
%%writefile outputs.tf
output "api_urls" {
  description = "where the replicas are listening"
  value       = [for c in docker_container.api : "http://127.0.0.1:${c.ports[0].external}"]
}

output "network" {
  value = docker_network.app.name
}

output "container_ids" {
  value     = [for c in docker_container.api : substr(c.id, 0, 12)]
  sensitive = false
}

Writing outputs.tf


Apply the stack.

In [18]:
tf("plan", "-no-color", quiet=True)
out = tf("apply", "-auto-approve", "-no-color")

mage.api will be created
  + resource "docker_image" "api" {
      + id           = (known after apply)
      + image_id     = (known after apply)
      + keep_locally = true
      + name         = "nginx:1.27-alpine"
      + repo_digest  = (known after apply)
    }

  # docker_network.app will be created
  + resource "docker_network" "app" {
      + driver      = (known after apply)
      + id          = (known after apply)
      + internal    = (known after apply)
      + ipam_driver = "default"
      + name        = "churn-net"
      + options     = (known after apply)
      + scope       = (known after apply)

      + ipam_config (known after apply)
    }

Plan: 4 to add, 0 to change, 0 to destroy.

Changes to Outputs:
  + api_urls      = [
      + "http://127.0.0.1:8082",
      + "http://127.0.0.1:8083",
    ]
  + container_ids = [
      + (known after apply),
      + (known after apply),
    ]
  + network       = "churn-net"
docker_network.app: Creating...
docker_image.api: Creat

Read the outputs back as JSON, then confirm with Docker directly.

In [19]:
print(tf("output", "-json", "-no-color", quiet=True))
!docker ps --filter label=APP --format 'table {{.Names}}\t{{.Ports}}' 2>/dev/null || docker ps --filter name=churn-api --format 'table {{.Names}}\t{{.Ports}}'

{
  "api_urls": {
    "sensitive": false,
    "type": [
      "tuple",
      [
        "string",
        "string"
      ]
    ],
    "value": [
      "http://127.0.0.1:8082",
      "http://127.0.0.1:8083"
    ]
  },
  "container_ids": {
    "sensitive": false,
    "type": [
      "tuple",
      [
        "string",
        "string"
      ]
    ],
    "value": [
      "564ed2c59116",
      "8c03cb6a7645"
    ]
  },
  "network": {
    "sensitive": false,
    "type": "string",
    "value": "churn-net"
  }
}

{.Names}   {.Ports}


The dependency graph Terraform worked out on its own from those references. Nobody drew this.

In [20]:
# the dependency graph Terraform worked out for itself
graph = tf("graph", "-no-color", quiet=True)
edges = [l.strip() for l in graph.splitlines() if "->" in l]
print(f"{len(edges)} edges in the graph, for example:")
for e in edges[:6]:
    print("  ", e.replace('"', ""))
print("\nYou never wrote an order. The references implied it, and Terraform")
print("creates the network before the containers because it has to.")

3 edges in the graph, for example:
   docker_container.api -> docker_image.api;
   docker_container.api -> docker_network.app;
   docker_container.web -> docker_image.nginx;

You never wrote an order. The references implied it, and Terraform
creates the network before the containers because it has to.


### Changing a variable changes the world

`replicas = 2` is not a comment; it is the desired state. Set it to 3 and Terraform adds one
container — it does not rebuild the other two.

In [21]:
out = tf("apply", "-auto-approve", "-no-color", "-var", "replicas=3")
!docker ps --filter name=churn-api --format '{{.Names}}' | sort
print("\n-- and back down again --")
out = tf("apply", "-auto-approve", "-no-color", "-var", "replicas=1")
!docker ps --filter name=churn-api --format '{{.Names}}' | sort
print("\nScaling up added one. Scaling down destroyed two. Neither touched the survivor.")

     = false
      + runtime                                     = (known after apply)
      + security_opts                               = (known after apply)
      + shm_size                                    = (known after apply)
      + start                                       = true
      + stdin_open                                  = false
      + stop_signal                                 = (known after apply)
      + stop_timeout                                = (known after apply)
      + tty                                         = false
      + wait                                        = false
      + wait_timeout                                = 60

      + healthcheck (known after apply)

      + labels (known after apply)

      + networks_advanced {
          + aliases      = []
          + name         = "churn-net"
            # (3 unchanged attributes hidden)
        }

      + ports {
          + external = 8084
          + internal = 80
          + ip     

---
# Part 5 — Advanced   ·   deck slides 16–20

## Step 5.1 · Drift — when someone changes things by hand

This is the failure mode that matters in a real team. Terraform's answer is that reality always
loses to the configuration.

In [22]:
# somebody "just fixes something quickly" outside Terraform
!docker rm -f tf-web >/dev/null 2>&1 && echo "container deleted by hand, behind Terraform's back"

out = tf("plan", "-no-color")
assert "1 to add" in out or "to add" in out, "the plan should notice the missing container"

container deleted by hand, behind Terraform's back
                     = 0
      + must_run                                    = true
      + name                                        = "tf-web"
      + network_data                                = (known after apply)
      + network_mode                                = "bridge"
      + read_only                                   = false
      + remove_volumes                              = true
      + restart                                     = "no"
      + rm                                          = false
      + runtime                                     = (known after apply)
      + security_opts                               = (known after apply)
      + shm_size                                    = (known after apply)
      + start                                       = true
      + stdin_open                                  = false
      + stop_signal                                 = (known after apply)
      + stop

Someone deletes a container by hand — the classic Friday incident. Terraform notices the **drift**
and puts it back.

In [23]:
out = tf("apply", "-auto-approve", "-no-color", quiet=True)
print(out[-300:])
!docker ps --filter name=tf-web --format 'table {{.Names}}\t{{.Status}}'
print("\nTerraform put it back. The configuration is the source of truth, not the cluster.")

e after 1s [id=8187b61996f7152348b168226eacebcb6fbd06026dc4bdc5726a325ea38acc74]

Apply complete! Resources: 2 added, 0 changed, 0 destroyed.

Outputs:

api_urls = [
  "http://127.0.0.1:8082",
  "http://127.0.0.1:8083",
]
container_ids = [
  "564ed2c59116",
  "8187b61996f7",
]
network = "churn-net"

{.Names}   {.Status}
{.Names}   {.Status}

Terraform put it back. The configuration is the source of truth, not the cluster.


## Step 5.2 · The rest of the toolkit

```hcl
# --- remote state: required the moment two people share a stack -------------
terraform {
  backend "s3" {
    bucket         = "acme-tfstate"
    key            = "churn/prod.tfstate"
    region         = "eu-west-1"
    dynamodb_table = "tf-locks"      # the LOCK is the point, as much as the storage
    encrypt        = true
  }
}

# --- a module: the unit of reuse --------------------------------------------
module "api" {
  source   = "./modules/service"
  name     = "churn"
  replicas = 3
}

# --- adopt something that already exists -----------------------------------
terraform import docker_container.web  <container-id>

# --- separate environments -------------------------------------------------
terraform workspace new staging
terraform apply -var-file=staging.tfvars
```

| Command | For |
|---|---|
| `terraform fmt` | canonical formatting; run it in CI |
| `terraform validate` | syntax and types, no cloud calls |
| `terraform plan -out=tf.plan` | save a plan, apply *exactly* that later |
| `terraform state mv` | rename a resource without recreating it |
| `terraform destroy -target=...` | remove one thing (use sparingly) |

## Step 5.3 · The habits that prevent incidents

| Habit | Why |
|---|---|
| **Read every plan.** Especially the `-/+` lines | that is destroy-and-recreate, on real data |
| `plan -out=tf.plan`, then `apply tf.plan` | the thing reviewed is the thing applied |
| Remote state with locking, from day two | two simultaneous applies corrupt state |
| Never hand-edit state | `state mv` / `rm` / `import` exist for this |
| `prevent_destroy` on databases | a lifecycle block is cheaper than a restore |
| Pin provider versions | a minor provider bump can plan a replacement |
| One stack per environment | a shared stack means staging can break production |

```hcl
resource "docker_container" "db" {
  lifecycle {
    prevent_destroy = true       # apply will fail rather than delete this
  }
}
```

---
# Reference — worth knowing, not demonstrated here

| Topic | One-line version |
|---|---|
| Cloud providers | `provider "aws"`; resource types change, workflow does not |
| Modules and the registry | `source = "terraform-aws-modules/vpc/aws"` |
| `for_each` vs `count` | `for_each` keys resources by name — safer when the list changes order |
| Data sources | read something you do not manage: `data "aws_ami" "x"` |
| CI/CD for Terraform | plan on pull request, apply on merge — session 11's shape |
| Terragrunt / Terraform Cloud | DRY multi-environment, and hosted state with policy checks |
| OpenTofu | the fork after the licence change; drop-in for most configurations |

## Recap

| You wanted to… | Do this |
|---|---|
| set up | `terraform init` (commit the lock file) |
| see what would happen | `terraform plan` — read the symbols |
| make it happen | `terraform apply` |
| know what exists | `terraform state list` / `show` |
| parameterise | `variable` blocks, `-var`, `.tfvars` |
| pass values out | `output` blocks |
| several copies | `count` or `for_each` |
| protect something | `lifecycle { prevent_destroy = true }` |
| fix hand-made changes | `terraform apply` — reality loses |
| adopt existing infrastructure | `terraform import` |
| work as a team | remote state **with locking** |

## What to do at work tomorrow

1. Pick the smallest piece of real infrastructure you own and write it as one resource.
2. `plan`, read it line by line, then `apply`.
3. Put the state in a remote backend **with locking** before a second person touches it.
4. Add `prevent_destroy` to anything holding data you cannot recreate.
5. Move `plan` into CI on pull requests. A reviewed plan is the entire value proposition.

## Cleanup

Destroys everything this notebook created, then removes the sandbox.

In [24]:
import shutil, os, subprocess
out = tf("destroy", "-auto-approve", "-no-color", check=False, quiet=True)
print(out[-300:])
for name in ("tf-web", "churn-api-0", "churn-api-1", "churn-api-2"):
    subprocess.run(["docker", "rm", "-f", name], capture_output=True)
print("infrastructure destroyed")

os.chdir(BASE)
shutil.rmtree(PROJ, ignore_errors=True)
print("sandbox removed")

77f41e938f9dd53dfccf89d066c02e3]
docker_image.api: Destroying... [id=sha256:6769dc3a703c719c1d2756bda113659be28ae16cf0da58dd5fd823d6b9a050eanginx:1.27-alpine]
docker_image.api: Destruction complete after 0s
docker_network.app: Destruction complete after 2s

Destroy complete! Resources: 6 destroyed.

infrastructure destroyed
sandbox removed
